# Stage 3 Wander — Boundary and Obstacle Avoidance

Camera tilts down at startup. Robot drives forward and avoids yellow tape and obstacles.  
No cap detection · No arm control · No Roboflow.

---
## Run Order and Safety

**Every time you open or re-open this notebook, run cells in this order:**

| Step | Cell |
|------|------|
| 1 | **Cleanup** (immediately below) |
| 2 | Imports |
| 3 | Configuration |
| 4 | Camera Tilt |
| 5 | Camera init |
| 6 | Robot init |
| 7 | Helper Functions |
| 8 | Detection functions + draw_debug |
| 9 | State machine globals + execute() |
| 10 | **Start Navigation** |

**If you get a runtime error at any step:**
> `Kernel → Restart Kernel` then run from the top.

**If the camera fails to initialize:**
> `Kernel → Shutdown All Kernels`, then reopen this notebook.

**Never run Start twice without running Stop/Reset first.**  
**Never run the Camera or Robot cells while navigation is active.**

---

In [ ]:
# ── CLEANUP — run this first, every time ────────────────────────────
# Safe to run even if nothing has been initialized yet.

running = False

try:
    camera.unobserve_all()
    camera.stop()
    print('Camera observers cleared.')
except Exception:
    pass

try:
    robot.left_motor.value  = 0.0
    robot.right_motor.value = 0.0
    print('Robot motors zeroed.')
except Exception:
    pass

# Always define these so downstream cells never get NameError
camera = None
robot  = None

print('Cleanup done. Safe to re-run all cells.')

## 1. Imports

In [2]:
import cv2
import numpy as np
import time
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, Robot, bgr8_to_jpeg

print('Imports OK')

Imports OK


In [3]:
try:
    from SCSCtrl import TTLServo
    _servo_available = True
    print('TTLServo loaded.')
except Exception as _e:
    _servo_available = False
    print('TTLServo not available:', _e)
    print('Camera tilt will be skipped.')

Succeeded to open the port
Succeeded to change the baudrate
TTLServo loaded.


## 2. Configuration

Tune `YELLOW_BOUNDARY_THRESHOLD` and `OBSTACLE_EDGE_THRESHOLD` using **Test 5** before full wander.  
`CAMERA_TILT_DOWN` — degrees to tilt down; colorTracking uses 25 as max, 15 is a safe default.

In [4]:
import yaml
import numpy as np
try:
    with open("config.yaml", encoding="utf-8") as f_cfg:
        _config = yaml.safe_load(f_cfg)
    CAMERA_WIDTH = _config["camera"]["width"]
    CAMERA_HEIGHT = _config["camera"]["height"]
    TAPE_HSV_LOWER = np.array(_config["color"]["lower_hsv"])
    TAPE_HSV_UPPER = np.array(_config["color"]["upper_hsv"])
    print("Loaded configuration from config.yaml")
except Exception as _e_cfg:
    print("Could not load config.yaml, using defaults:", _e_cfg)
    CAMERA_WIDTH  = 320
    CAMERA_HEIGHT = 240
    TAPE_HSV_LOWER = np.array([15,  60,  60])
    TAPE_HSV_UPPER = np.array([45, 255, 255])

# ── Camera tilt ──────────────────────────────────────────────────────
# Servo 1 = PAN (horizontal).  Servo 5 = TILT (up/down).
CAMERA_TILT_DOWN = 15   # degrees; safe range 0-25

# ── Motor speeds ─────────────────────────────────────────────────────
FORWARD_SPEED = 0.18
REVERSE_SPEED = 0.15
TURN_SPEED    = 0.22
MAX_SPEED     = 0.25

# ── Avoidance timing ─────────────────────────────────────────────────
AVOID_REVERSE_TIME = 0.40
AVOID_TURN_TIME    = 0.55
WANDER_COOLDOWN    = 0.50

# ── Yellow tape ROI ──────────────────────────────────────────────────
YELLOW_ROI_START = 0.65

# ── Obstacle ROI ─────────────────────────────────────────────────────
FRONT_ROI_TOP_FRAC    = 0.15
FRONT_ROI_BOTTOM_FRAC = 0.60

# ── Detection thresholds ─────────────────────────────────────────────
YELLOW_BOUNDARY_THRESHOLD = 1800
OBSTACLE_EDGE_THRESHOLD   = 500
OBSTACLE_EQUAL_THRESH     = 50

# ── Canny ─────────────────────────────────────────────────────────────
CANNY_LOW  = 40
CANNY_HIGH = 120

print("Configuration loaded.")


Loaded configuration from config.yaml
Configuration loaded.


## 3. Camera Tilt

**Servo 1** = PAN (horizontal).  **Servo 5** = TILT (up/down) — adjusted here to see more floor.  
Skipped safely if the servo library is unavailable.

In [5]:
if _servo_available:
    try:
        TTLServo.servoAngleCtrl(1, 0, 1, 100)                 # center pan
        time.sleep(0.15)
        TTLServo.servoAngleCtrl(5, CAMERA_TILT_DOWN, 1, 100)  # tilt down
        time.sleep(0.50)
        print('Camera: pan centered, tilt down', CAMERA_TILT_DOWN, 'deg')
    except Exception as e:
        print('Servo init error:', e)
else:
    print('Servo not available — camera tilt skipped.')

Camera: pan centered, tilt down 15 deg


## 4. Camera and Robot

Both cells are defensive: they print clear errors instead of crashing,  
and safe to re-run in the same kernel session.

In [ ]:
import threading, subprocess as _sp

# Clean up any stale camera first
try:
    camera.unobserve_all()
    camera.stop()
except Exception:
    pass

camera       = None
_camera_mode = None


class _OpenCVCamera:
    """Thread-based camera shim that mimics the JetBot Camera API."""
    def __init__(self, device=0, backend=None, width=300, height=300):
        if backend is None:
            self._cap = cv2.VideoCapture(device)
        else:
            self._cap = cv2.VideoCapture(device, backend)
        if not self._cap.isOpened():
            raise RuntimeError('VideoCapture failed to open: ' + str(device))
        self._lock    = threading.Lock()
        self._running = True
        self._cbs     = []
        self._width   = width
        self._height  = height
        self.value    = np.zeros((height, width, 3), dtype=np.uint8)
        self._thread  = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def _loop(self):
        while self._running:
            ok, frame = self._cap.read()
            if ok:
                frame = cv2.resize(frame, (self._width, self._height))
                with self._lock:
                    self.value = frame
                ch = {'new': frame}
                for cb in list(self._cbs):
                    try:
                        cb(ch)
                    except Exception:
                        pass

    def observe(self, callback, names='value'):
        if callback not in self._cbs:
            self._cbs.append(callback)

    def unobserve_all(self):
        self._cbs.clear()

    def stop(self):
        self._running = False
        try:
            self._cap.release()
        except Exception:
            pass


def _recover_nvargus():
    try:
        _sp.run(['fuser', '-k', '/dev/video0'], capture_output=True, timeout=3)
        _sp.run(['pkill', '-9', 'nvargus-daemon'], capture_output=True, timeout=3)
        _sp.Popen(['nvargus-daemon'], stdout=_sp.DEVNULL, stderr=_sp.DEVNULL)
        time.sleep(2.5)
        print('  nvargus-daemon restarted.')
    except Exception as _e:
        print('  nvargus recovery error:', _e)


# ── Attempt 1: JetBot/GStreamer ───────────────────────────────────────
print('Camera: Attempt 1 — JetBot/GStreamer ...')
try:
    camera       = Camera.instance(width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
    _camera_mode = 'jetbot'
    print('  Ready.  Mode: jetbot/GStreamer')
except Exception as _e1:
    print('  Failed:', _e1)

    # ── Attempt 2: nvargus recovery + retry ──────────────────────────
    print('Camera: Attempt 2 — nvargus recovery + retry ...')
    try:
        _recover_nvargus()
        camera       = Camera.instance(width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
        _camera_mode = 'jetbot'
        print('  Ready after recovery.  Mode: jetbot/GStreamer')
    except Exception as _e2:
        print('  Failed:', _e2)

        # ── Attempt 3: OpenCV VideoCapture ───────────────────────────
        print('Camera: Attempt 3 — OpenCV VideoCapture(0) ...')
        try:
            camera       = _OpenCVCamera(device=0,
                                         width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
            _camera_mode = 'opencv'
            print('  Ready.  Mode: OpenCV')
        except Exception as _e3:
            print('  Failed:', _e3)

            # ── Attempt 4: V4L2 direct ────────────────────────────────
            print('Camera: Attempt 4 — V4L2 direct /dev/video0 ...')
            try:
                camera       = _OpenCVCamera(device='/dev/video0',
                                             backend=cv2.CAP_V4L2,
                                             width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
                _camera_mode = 'v4l2'
                print('  Ready.  Mode: V4L2 direct')
            except Exception as _e4:
                print('  Failed:', _e4)
                camera       = None
                _camera_mode = None
                print()
                print('ALL camera attempts failed.')
                print('In Docker, run in a host terminal or restart the container:')
                print('  fuser -k /dev/video0')
                print('  pkill -9 nvargus-daemon')
                print('Then restart the container and reopen this notebook.')

if camera is not None:
    print('Camera ready.  Mode:', _camera_mode)

In [6]:
# Defensive robot init
try:
    robot.stop()
except Exception:
    pass

try:
    robot = Robot()
    robot.stop()
    print('Robot ready.')
except Exception as e:
    robot = None
    print('ERROR: Robot initialization failed.')
    print('  Reason:', e)
    print()
    print('  Fix: Kernel -> Restart Kernel, then run from the top.')

Robot ready.


## 5. Helper Functions

In [7]:
def clamp(val, lo, hi):
    if val < lo: return lo
    if val > hi: return hi
    return val


def safe_stop():
    """Zero motors without raising. Safe to call when robot is None."""
    try:
        robot.left_motor.value  = 0.0
        robot.right_motor.value = 0.0
    except Exception:
        pass


def set_drive(left, right):
    """Drive motors, clamped. No-op if robot is not initialized."""
    try:
        robot.left_motor.value  = clamp(left,  -MAX_SPEED, MAX_SPEED)
        robot.right_motor.value = clamp(right, -MAX_SPEED, MAX_SPEED)
    except Exception:
        pass


print('clamp / safe_stop / set_drive defined.')

clamp / safe_stop / set_drive defined.


## 6. Yellow Boundary Detection

Only checks frame rows **below `YELLOW_ROI_START`** (default 65 % → bottom 35 %).  
Yellow tape higher in the frame is ignored.

In [8]:
def detect_yellow_boundary(frame):
    h         = frame.shape[0]
    roi_start = int(h * YELLOW_ROI_START)
    roi       = frame[roi_start:, :]

    hsv_roi  = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    roi_mask = cv2.inRange(hsv_roi, TAPE_HSV_LOWER, TAPE_HSV_UPPER)

    yellow_area  = int(np.sum(roi_mask > 0))
    mid          = roi_mask.shape[1] // 2
    yellow_left  = int(np.sum(roi_mask[:, :mid] > 0))
    yellow_right = int(np.sum(roi_mask[:, mid:] > 0))

    full_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    full_mask[roi_start:, :] = roi_mask

    return {
        'detected':     yellow_area >= YELLOW_BOUNDARY_THRESHOLD,
        'yellow_area':  yellow_area,
        'yellow_left':  yellow_left,
        'yellow_right': yellow_right,
        'yellow_mask':  full_mask,
    }


print('detect_yellow_boundary() defined.')

detect_yellow_boundary() defined.


## 7. Obstacle Detection

Checks rows **15 – 60 %** of frame height. Yellow pixels masked out first.  
Turn direction: more edges left → turn right; more right → turn left; equal → alternate.

In [9]:
def detect_obstacle(frame):
    h, w    = frame.shape[:2]
    roi_top = int(h * FRONT_ROI_TOP_FRAC)
    roi_bot = int(h * FRONT_ROI_BOTTOM_FRAC)
    roi     = frame[roi_top:roi_bot, :]

    hsv_roi     = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    yellow_mask = cv2.inRange(hsv_roi, TAPE_HSV_LOWER, TAPE_HSV_UPPER)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray[yellow_mask > 0] = 0

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges   = cv2.Canny(blurred, CANNY_LOW, CANNY_HIGH)
    edges   = cv2.dilate(edges, None, iterations=1)

    mid        = edges.shape[1] // 2
    edge_left  = int(np.sum(edges[:, :mid] > 0))
    edge_right = int(np.sum(edges[:, mid:] > 0))
    edge_total = edge_left + edge_right

    full_mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    full_mask[roi_top:roi_bot, :] = edges

    return {
        'detected':   edge_total >= OBSTACLE_EDGE_THRESHOLD,
        'edge_total': edge_total,
        'edge_left':  edge_left,
        'edge_right': edge_right,
        'mask':       full_mask,
    }


print('detect_obstacle() defined.')

detect_obstacle() defined.


## 8. Debug Visualization

Left widget — annotated frame:
- **Cyan horizontal line** at `YELLOW_ROI_START` — yellow detection zone starts here
- **Orange rectangle** — obstacle detection zone (turns red when firing)
- Text: state `S`, motors `L/R`, yellow area `Y`, edge count `E`, turn dir `T`

Right widget — **cyan** = yellow tape pixels, **orange** = obstacle edges

In [10]:
def draw_debug(frame, boundary, obstacle, state, left_spd, right_spd, turn_dir):
    out  = frame.copy()
    h, w = out.shape[:2]

    # Yellow ROI: horizontal line + light tint below it
    roi_y   = int(h * YELLOW_ROI_START)
    y_color = (0, 0, 255) if boundary['detected'] else (0, 220, 220)
    overlay = out.copy()
    cv2.rectangle(overlay, (0, roi_y), (w - 1, h - 1), y_color, -1)
    cv2.addWeighted(overlay, 0.12, out, 0.88, 0, out)
    cv2.line(out, (0, roi_y), (w - 1, roi_y), y_color, 2)

    # Obstacle ROI rectangle
    obs_top = int(h * FRONT_ROI_TOP_FRAC)
    obs_bot = int(h * FRONT_ROI_BOTTOM_FRAC)
    o_color = (0, 0, 255) if obstacle['detected'] else (255, 130, 0)
    cv2.rectangle(out, (0, obs_top), (w - 1, obs_bot), o_color, 2)

    # Centerline
    cv2.line(out, (w // 2, 0), (w // 2, h), (70, 70, 70), 1)

    # Text
    cv2.putText(out, 'S:' + state,
                (5, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1)
    cv2.putText(out,
                'L=' + str(round(left_spd, 2)) + ' R=' + str(round(right_spd, 2)),
                (5, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)
    cv2.putText(out, 'Y=' + str(boundary['yellow_area']),
                (5, 49), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 240, 240), 1)
    cv2.putText(out, 'E=' + str(obstacle['edge_total']),
                (5, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 130, 0), 1)
    cv2.putText(out, 'T:' + turn_dir,
                (5, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (200, 200, 0), 1)

    if boundary['detected']:
        cv2.putText(out, 'YELLOW!',
                    (5, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (0, 0, 255), 2)
    elif obstacle['detected']:
        cv2.putText(out, 'OBSTACLE!',
                    (5, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (0, 80, 255), 2)

    return out


print('draw_debug() defined.')

draw_debug() defined.


## 9. State Machine

| State | Behaviour |
|-------|-----------|
| `WANDER` | Drive forward. Yellow checked first (higher priority), then obstacle. |
| `AVOID_YELLOW` | Phase 1: reverse. Phase 2: turn away from yellow-heavy side. Phase 3: `WANDER`. |
| `AVOID_OBSTACLE` | Same phases. Turn toward the emptier side; alternate if equal. |
| `STOPPED` | Emergency only. No auto-recovery. |

No blocking `sleep()` inside the callback — all timing uses timestamps.

In [11]:
nav_state           = 'WANDER'
avoid_start_time    = 0.0
avoid_turn_right    = True
_obstacle_alt_right = True
wander_cooldown_end = 0.0
last_left_spd       = 0.0
last_right_spd      = 0.0
running             = False   # set True only by Start cell

print('State machine globals defined.  running =', running)

State machine globals defined.  running = False


In [12]:
def execute(change):
    global nav_state, avoid_start_time, avoid_turn_right
    global _obstacle_alt_right, wander_cooldown_end
    global last_left_spd, last_right_spd

    # Guard: ignore frames if navigation has been stopped
    if not running:
        return

    frame     = change['new']
    left_spd  = 0.0
    right_spd = 0.0

    try:
        boundary = detect_yellow_boundary(frame)
        obstacle = detect_obstacle(frame)
        now      = time.time()

        # ── State machine ────────────────────────────────────────

        if nav_state == 'STOPPED':
            pass

        elif nav_state in ('AVOID_YELLOW', 'AVOID_OBSTACLE'):
            elapsed = now - avoid_start_time
            if elapsed < AVOID_REVERSE_TIME:
                left_spd  = -REVERSE_SPEED
                right_spd = -REVERSE_SPEED
            elif elapsed < AVOID_REVERSE_TIME + AVOID_TURN_TIME:
                if avoid_turn_right:
                    left_spd  =  TURN_SPEED
                    right_spd = -TURN_SPEED
                else:
                    left_spd  = -TURN_SPEED
                    right_spd =  TURN_SPEED
            else:
                wander_cooldown_end = now + WANDER_COOLDOWN
                nav_state = 'WANDER'
                left_spd  = FORWARD_SPEED
                right_spd = FORWARD_SPEED

        elif nav_state == 'WANDER':
            if now < wander_cooldown_end:
                left_spd  = FORWARD_SPEED
                right_spd = FORWARD_SPEED

            elif boundary['detected']:
                avoid_turn_right = boundary['yellow_left'] >= boundary['yellow_right']
                avoid_start_time = now
                nav_state        = 'AVOID_YELLOW'
                left_spd         = -REVERSE_SPEED
                right_spd        = -REVERSE_SPEED

            elif obstacle['detected']:
                el = obstacle['edge_left']
                er = obstacle['edge_right']
                if el - er > OBSTACLE_EQUAL_THRESH:
                    avoid_turn_right = True    # more on left  -> turn right
                elif er - el > OBSTACLE_EQUAL_THRESH:
                    avoid_turn_right = False   # more on right -> turn left
                else:
                    avoid_turn_right    = _obstacle_alt_right
                    _obstacle_alt_right = not _obstacle_alt_right
                avoid_start_time = now
                nav_state        = 'AVOID_OBSTACLE'
                left_spd         = -REVERSE_SPEED
                right_spd        = -REVERSE_SPEED

            else:
                left_spd  = FORWARD_SPEED
                right_spd = FORWARD_SPEED

        # ── Drive ────────────────────────────────────────────────
        set_drive(left_spd, right_spd)
        last_left_spd  = left_spd
        last_right_spd = right_spd

        # ── Debug display ────────────────────────────────────────
        td                 = 'RIGHT' if avoid_turn_right else 'LEFT'
        annotated          = draw_debug(frame, boundary, obstacle,
                                        nav_state, left_spd, right_spd, td)
        image_widget.value = bgr8_to_jpeg(annotated)

        combined = np.zeros((CAMERA_HEIGHT, CAMERA_WIDTH, 3), dtype=np.uint8)
        ym = boundary['yellow_mask']
        om = obstacle['mask']
        combined[ym > 0] = [0, 255, 255]
        combined[om > 0] = [0, 140, 255]
        mask_widget.value = bgr8_to_jpeg(combined)

        state_label.value  = 'State: ' + nav_state
        motor_label.value  = ('Motors L=' + str(round(left_spd, 3)) +
                              '  R=' + str(round(right_spd, 3)))
        yellow_label.value = ('Yellow: ' + str(boundary['yellow_area']) +
                              '  L=' + str(boundary['yellow_left']) +
                              '  R=' + str(boundary['yellow_right']))
        obs_label.value    = ('Edges: ' + str(obstacle['edge_total']) +
                              '  L=' + str(obstacle['edge_left']) +
                              '  R=' + str(obstacle['edge_right']))
        turn_label.value   = 'Turn dir: ' + td

    except Exception as e:
        safe_stop()
        print('[ERROR in execute] ' + str(e))
        raise


print('execute() defined.')

execute() defined.


In [ ]:
# ── Display widgets — run once, leave visible ────────────────────────
image_widget = widgets.Image(format='jpeg', width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
mask_widget  = widgets.Image(format='jpeg', width=CAMERA_WIDTH, height=CAMERA_HEIGHT)
state_label  = widgets.Label(value='State: STOPPED')
motor_label  = widgets.Label(value='Motors: L=0.00  R=0.00')
yellow_label = widgets.Label(value='Yellow: 0  L=0  R=0')
obs_label    = widgets.Label(value='Edges: 0  L=0  R=0')
turn_label   = widgets.Label(value='Turn dir: RIGHT')

display(widgets.VBox([
    widgets.HBox([image_widget, mask_widget]),
    state_label,
    motor_label,
    yellow_label,
    obs_label,
    turn_label,
]))
print('Display widgets ready.')

## 10. Start Navigation

Guards: will not start if camera or robot failed to initialize.  
Calls `unobserve_all()` first to prevent duplicate callbacks.

In [ ]:
try:
    _cam = camera
except NameError:
    _cam = None

try:
    _rob = robot
except NameError:
    _rob = None

if _cam is None:
    print('ERROR: Camera not initialized. Run the Camera cell (Section 4) first.')
elif _rob is None:
    print('ERROR: Robot not initialized. Run the Robot cell (Section 4) first.')
else:
    nav_state           = 'WANDER'
    avoid_start_time    = 0.0
    avoid_turn_right    = True
    _obstacle_alt_right = True
    wander_cooldown_end = 0.0
    last_left_spd       = 0.0
    last_right_spd      = 0.0

    camera.unobserve_all()
    camera.observe(execute, names='value')
    running = True

    try:
        state_label.value = 'State: WANDER'
    except NameError:
        pass

    print('Navigation started.  State:', nav_state)

## 11. Emergency Stop

Safe to run even if camera or robot is not initialized.

In [ ]:
running = False

try:
    camera.unobserve_all()
    print('Camera observers cleared.')
except Exception:
    pass

try:
    robot.stop()
    print('Robot stopped.')
except Exception:
    pass

nav_state = 'STOPPED'

try:
    state_label.value = 'State: STOPPED'
    motor_label.value = 'Motors: L=0.00  R=0.00'
except Exception:
    pass

print('EMERGENCY STOP done.')

## 12. Reset and Restart

After stopping, reset state and restart without re-running all cells.  
Safe to call even if objects are in an unknown state.

In [ ]:
running = False

try:
    camera.unobserve_all()
except Exception:
    pass

try:
    robot.stop()
except Exception:
    pass

try:
    _cam = camera
    _rob = robot
except NameError:
    _cam = None
    _rob = None

if _cam is None or _rob is None:
    print('ERROR: camera or robot not initialized — run the init cells first.')
else:
    nav_state           = 'WANDER'
    avoid_start_time    = 0.0
    avoid_turn_right    = True
    _obstacle_alt_right = True
    wander_cooldown_end = 0.0
    last_left_spd       = 0.0
    last_right_spd      = 0.0

    camera.observe(execute, names='value')
    running = True

    try:
        state_label.value = 'State: WANDER'
        motor_label.value = 'Motors: L=0.00  R=0.00'
    except Exception:
        pass

    print('Reset to WANDER. Navigation restarted.')

## 13. Test Procedures

Run these **before** starting the full wander to verify each subsystem.

---

### Test 1 — Yellow Boundary Detection (static)

Hold yellow tape below the camera or position the robot near the arena edge.

```python
frame  = camera.value.copy()
result = detect_yellow_boundary(frame)
print('detected:', result['detected'],
      ' area:', result['yellow_area'],
      ' L:', result['yellow_left'],
      ' R:', result['yellow_right'])

vis   = frame.copy()
h     = vis.shape[0]
roi_y = int(h * YELLOW_ROI_START)
cv2.line(vis, (0, roi_y), (vis.shape[1]-1, roi_y), (0, 220, 220), 2)
mask_bgr = cv2.cvtColor(result['yellow_mask'], cv2.COLOR_GRAY2BGR)
display(widgets.Image(value=bgr8_to_jpeg(vis),      format='jpeg'))
display(widgets.Image(value=bgr8_to_jpeg(mask_bgr), format='jpeg'))
```

**Pass:** `detected` is `True` with tape visible, `False` on clean floor.  
Adjust `YELLOW_BOUNDARY_THRESHOLD` if needed.

---

### Test 2 — Obstacle Detection (static)

Hold a cardboard box ~30 cm in front of the robot.

```python
frame  = camera.value.copy()
result = detect_obstacle(frame)
print('detected:', result['detected'],
      ' edges:', result['edge_total'],
      ' L:', result['edge_left'],
      ' R:', result['edge_right'])

edge_bgr = cv2.cvtColor(result['mask'], cv2.COLOR_GRAY2BGR)
display(widgets.Image(value=bgr8_to_jpeg(edge_bgr), format='jpeg'))
```

**Pass:** `detected` is `True` with box present, `False` on clear floor.  
Adjust `OBSTACLE_EDGE_THRESHOLD` or `CANNY_LOW/HIGH` if floor triggers false positives.

---

### Test 3 — Motor Directions (robot on blocks)

Lift the robot so tracks spin freely.

```python
import time
print('Forward 1 s')
set_drive(FORWARD_SPEED,  FORWARD_SPEED);  time.sleep(1.0); safe_stop()
time.sleep(0.5)
print('Reverse 1 s')
set_drive(-REVERSE_SPEED, -REVERSE_SPEED); time.sleep(1.0); safe_stop()
time.sleep(0.5)
print('Turn right 1 s')
set_drive(TURN_SPEED,     -TURN_SPEED);    time.sleep(1.0); safe_stop()
```

**Pass:** tracks spin in the expected directions.

---

### Test 4 — Full Wander (robot in arena)

Run **Section 10 — Start Navigation**.

**Expected:**
1. Robot drives forward slowly in `WANDER`.
2. Yellow tape in bottom ROI → `AVOID_YELLOW`: reverse, turn, back to `WANDER`.
3. Box in front ROI → `AVOID_OBSTACLE`: reverse, turn toward empty side, back to `WANDER`.
4. Cycle repeats indefinitely.

If the robot turns the wrong way on an obstacle, check `obs_label` for `edge_left` vs `edge_right`  
and adjust `OBSTACLE_EQUAL_THRESH`.

---

### Test 5 — Threshold Calibration (stationary, clear arena)

```python
frame = camera.value.copy()
b = detect_yellow_boundary(frame)
o = detect_obstacle(frame)
print('Baseline — yellow:', b['yellow_area'], '  edges:', o['edge_total'])
```

Set `YELLOW_BOUNDARY_THRESHOLD` to roughly **3× baseline yellow area**.  
Set `OBSTACLE_EDGE_THRESHOLD` to roughly **3× baseline edge count**.